# Challenge AI Engineer

## Daftar Isi

- **Soal 1B**
- **Soal 2**
- **Soal 3**

## Highlights

- Implemented a CLI-style FAQ chatbot using Ollama Cloud models, added model selection so the user can choose which cloud model to use.


## Soal 1B


In [ ]:
!pip -q install ollama requests


In [ ]:
import re
from difflib import SequenceMatcher
from pathlib import Path

import requests
from ollama import Client

print('Library siap digunakan.')


### OLLAMA API Key


In [ ]:
OLLAMA_API_KEY = "paste_your_key_here"
OLLAMA_HEADERS = {"Authorization": f"Bearer {OLLAMA_API_KEY}"}
client = Client(host='https://ollama.com', headers=OLLAMA_HEADERS)
print('API key terset.')


In [ ]:
def fetch_cloud_models():
    response = requests.get('https://ollama.com/api/tags', headers=OLLAMA_HEADERS, timeout=30)
    response.raise_for_status()
    payload = response.json()
    models = []
    for item in payload.get('models', []):
        name = item.get('name')
        if name and name not in models:
            models.append(name)
    return models


def is_subscription_error(exc):
    message = str(exc).lower()
    return 'requires a subscription' in message or 'upgrade for access' in message or 'status code: 403' in message


def probe_model_access(model_name):
    try:
        client.chat(model=model_name, messages=[{'role': 'user', 'content': 'Reply with OK.'}], stream=False)
        return True, None
    except Exception as exc:
        if is_subscription_error(exc):
            return False, 'subscription'
        return False, str(exc)


def discover_free_cloud_models():
    all_models = fetch_cloud_models()
    free_models = []
    blocked_models = []
    for model_name in all_models:
        ok, reason = probe_model_access(model_name)
        if ok:
            free_models.append(model_name)
        else:
            blocked_models.append((model_name, reason))
    return free_models, blocked_models


available_models, blocked_models = discover_free_cloud_models()
if not available_models:
    raise RuntimeError('Tidak ada model cloud yang bisa dipakai pada akun ini.')


def choose_model(prompt=None, default_model=None):
    if prompt is None:
        return default_model or available_models[0]
    choice = input(f'\n{prompt}').strip()
    if choice.startswith('/'):
        choice = choice[1:].strip()
    if choice.isdigit() and 1 <= int(choice) <= len(available_models):
        return available_models[int(choice) - 1]
    if choice:
        return choice
    return default_model or available_models[0]


default_model = 'gpt-oss:120b'
active_model = default_model if default_model in available_models else available_models[0]
print(f'Model default aktif: {active_model}')


In [ ]:
faq_path = Path('faq.txt')
if not faq_path.exists():
    raise FileNotFoundError('faq.txt tidak ditemukan. Upload file itu ke folder kerja notebook.')
print(f'faq.txt loaded from: {faq_path.resolve()}')


In [ ]:
def load_faq(path='faq.txt'):
    pairs = []
    current_q = None
    current_a = None
    for line in Path(path).read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if line.startswith('Q:'):
            current_q = line[2:].strip()
        elif line.startswith('A:'):
            current_a = line[2:].strip()
        if current_q and current_a:
            pairs.append((current_q, current_a))
            current_q = None
            current_a = None
    return pairs


def normalize(text):
    return re.findall(r'[a-z0-9]+', text.lower())


def similarity(a, b):
    ta = set(normalize(a))
    tb = set(normalize(b))
    if not ta or not tb:
        return 0.0
    jaccard = len(ta & tb) / len(ta | tb)
    ratio = SequenceMatcher(None, a.lower(), b.lower()).ratio()
    return 0.7 * jaccard + 0.3 * ratio


def best_faq_match(question, faq_pairs):
    scored = []
    for faq_question, faq_answer in faq_pairs:
        scored.append((similarity(question, faq_question), faq_question, faq_answer))
    scored.sort(reverse=True, key=lambda item: item[0])
    return scored[0] if scored else (0.0, '', '')


faq_pairs = load_faq()
print(f'FAQ loaded: {len(faq_pairs)} item')


In [ ]:
FALLBACK_ANSWER = 'Maaf, saya tidak dapat membantu dengan pertanyaan itu.'
MAX_CONTEXT_MESSAGES = 8

SYSTEM_PROMPT = (
    'Anda adalah chatbot FAQ Piala Dunia FIFA 2026. '
    'Jawab hanya berdasarkan konteks FAQ yang diberikan dan riwayat percakapan yang relevan. '
    'Jangan menambahkan fakta baru, jangan berasumsi, dan jangan menjawab di luar konteks. '
    f'Jika konteks tidak cukup, jawab persis: {FALLBACK_ANSWER}'
)

conversation = []


def build_messages(user_question, matched_question, matched_answer):
    faq_context = (
        'Konteks FAQ yang relevan:\n'
        f'Pertanyaan FAQ: {matched_question}\n'
        f'Jawaban FAQ: {matched_answer}\n\n'
        f'Pertanyaan pengguna: {user_question}\n'
        'Jawab singkat dan hanya berdasarkan jawaban FAQ di atas.'
    )
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT + '\n\n' + faq_context}]
    messages.extend(conversation[-MAX_CONTEXT_MESSAGES:])
    messages.append({'role': 'user', 'content': user_question})
    return messages


def ask_model(user_question):
    score, matched_question, matched_answer = best_faq_match(user_question, faq_pairs)
    if score < 0.20:
        return FALLBACK_ANSWER

    try:
        response = client.chat(
            model=active_model,
            messages=build_messages(user_question, matched_question, matched_answer),
            stream=False,
        )
        text = response.get('message', {}).get('content', '').strip()
        return text or matched_answer
    except Exception as exc:
        if is_subscription_error(exc):
            print(f'\n[Model {active_model} tidak tersedia karena subscription. Silakan ganti model.]')
            return FALLBACK_ANSWER
        print(f'\n[Fallback karena error Ollama Cloud: {exc}]')
        return matched_answer


def show_history():
    if not conversation:
        print('Belum ada riwayat percakapan.')
        return
    print('\nRiwayat konteks percakapan terakhir:')
    for item in conversation[-MAX_CONTEXT_MESSAGES:]:
        print(f"- {item['role']}: {item['content']}")


def switch_model():
    global active_model
    active_model = choose_model('Ketik nama model baru: ', active_model)
    print(f'Model aktif sekarang: {active_model}')


def run_chatbot():
    print('\nChatbot FAQ siap digunakan.')
    print(f'Model aktif: {active_model}')
    print('Ketik /model, /history, /reset, exit, atau quit.')

    while True:
        user_input = input('\nUser: ').strip()
        if not user_input:
            print('Bot: Silakan masukkan pertanyaan.')
            continue

        lowered = user_input.lower()
        if lowered in {'exit', 'quit'}:
            print('Bot: Terima kasih. Sesi chatbot selesai.')
            break
        if lowered == '/model':
            switch_model()
            continue
        if lowered == '/history':
            show_history()
            continue
        if lowered == '/reset':
            conversation.clear()
            print('Bot: Riwayat percakapan telah dihapus.')
            continue

        print('Bot: ', end='', flush=True)
        response = ask_model(user_input)
        print(response)
        conversation.append({'role': 'user', 'content': user_input})
        conversation.append({'role': 'assistant', 'content': response})


run_chatbot()


## Bagian 2 - Teori AI

### Soal 2: Pertanyaan Teori Dasar

**a. Apa yang dimaksud dengan Artificial Intelligence (AI)? Sebutkan dua contohnya dalam kehidupan sehari-hari.**

Artificial Intelligence (AI) adalah bidang ilmu komputer yang membuat mesin mampu meniru kemampuan cerdas manusia, seperti mengenali pola, memahami bahasa, mengambil keputusan, dan belajar dari data.

Dua contoh AI dalam kehidupan sehari-hari:

- Rekomendasi video di YouTube atau Netflix.
- Asisten virtual seperti Siri, Google Assistant, atau ChatGPT.

**b. Apa perbedaan antara Supervised Learning dan Unsupervised Learning? Berikan satu contoh untuk masing-masing.**

- **Supervised Learning** menggunakan data yang sudah memiliki label jawaban. Model belajar dari pasangan input-output yang benar. Contoh: klasifikasi email spam dan bukan spam.
- **Unsupervised Learning** menggunakan data tanpa label. Model mencari pola atau struktur sendiri. Contoh: clustering pelanggan berdasarkan perilaku belanja.

### Soal 3: Pertanyaan Konsep

**a. Apa itu Feature dalam konteks machine learning? Mengapa penting untuk memilih fitur yang tepat saat membangun model?**

Feature adalah variabel atau atribut yang digunakan model sebagai masukan untuk mempelajari pola. Contohnya umur, pendapatan, atau jumlah klik.

Pemilihan fitur yang tepat penting karena fitur yang relevan membantu model belajar lebih akurat, lebih cepat, dan lebih stabil. Fitur yang kurang tepat dapat membuat model sulit belajar atau menghasilkan prediksi yang kurang baik.

**b. Apa itu Fine-tuning dalam machine learning? Sebutkan satu kasus di mana fine-tuning berguna.**

Fine-tuning adalah proses menyesuaikan model yang sudah pre-trained agar lebih cocok dengan tugas atau data yang lebih spesifik.

Contoh kasus yang berguna: model bahasa umum di-fine-tune untuk klasifikasi sentimen ulasan pelanggan pada domain e-commerce atau layanan keuangan.
